# Statistics for Machine Learning — Notebook 6  
# Feature Engineering Using Statistics

## Why this notebook matters

Feature engineering is where statistics becomes practical.

A machine learning model does not understand raw data like humans do.  
It learns from numeric patterns.

Statistics helps us transform messy raw data into useful model features.

Examples:

- scale features so one column does not dominate
- log-transform skewed values
- cap outliers
- create bins from continuous values
- encode categories
- remove low-variance features
- remove highly correlated redundant features
- select useful features statistically
- detect data leakage
- build clean preprocessing pipelines

---

## What you will learn

1. What feature engineering means  
2. Why statistics guides preprocessing  
3. Missing value imputation  
4. Outlier capping using IQR and percentiles  
5. Log transform for skewed data  
6. Standardization and normalization  
7. Robust scaling  
8. Binning continuous variables  
9. One-hot encoding  
10. Target encoding intuition  
11. Low-variance feature removal  
12. Correlation filtering  
13. Chi-square feature selection  
14. ANOVA F-test feature selection  
15. Mutual information feature selection  
16. Leakage detection  
17. Full preprocessing pipeline  
18. Model comparison before and after feature engineering  
19. Mini project: complete statistical preprocessing pipeline

# 1. What is feature engineering?

Feature engineering means creating, transforming, selecting, or cleaning input variables so a model can learn better.

Raw data is rarely ready for machine learning.

Feature engineering can include:

- handling missing values
- converting categories to numbers
- scaling numeric variables
- transforming skewed data
- creating new useful columns
- removing noisy columns
- selecting important features

---

## Simple example

Raw feature:

```text
income = 150000
```

Engineered features:

```text
log_income = log(1 + income)
income_group = high
income_scaled = standardized value
```

Each transformation gives the model a different way to understand the same information.

# 2. Why statistics is important for feature engineering

Statistics tells us what transformation is suitable.

| Statistical signal | Possible action |
|---|---|
| Missing values | impute |
| Large scale differences | scale |
| Right skew | log transform |
| Outliers | cap or robust scale |
| Low variance | remove |
| High correlation | remove redundant feature |
| Category with many levels | encoding strategy |
| Feature strongly related to target | keep/select |
| Feature too perfectly related to target | check leakage |
| Nonlinear pattern | create bins or interactions |

A good AI engineer does not apply transformations blindly.  
They inspect the data first.

# 3. Setup

Run this cell first.

If something is missing:

```python
!pip install numpy pandas matplotlib seaborn scikit-learn scipy ipywidgets
```

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, KBinsDiscretizer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, SelectKBest, chi2, f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

try:
    import ipywidgets as widgets
    from ipywidgets import interact
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. Install it using: pip install ipywidgets")

np.random.seed(42)

%matplotlib inline
sns.set_theme(style="whitegrid")

print("Setup complete.")

# 4. Create a realistic messy ML dataset

We will create a customer churn dataset with realistic issues:

- numeric features on different scales
- skewed features
- outliers
- missing values
- categorical features
- redundant correlated features
- low-variance feature
- possible leakage feature

Target:

```text
churn = 1 means customer left
churn = 0 means customer stayed
```

In [ ]:
n = 3000

age = np.random.normal(36, 11, n).clip(18, 75)
income = np.random.lognormal(mean=10.7, sigma=0.55, size=n).clip(12000, 250000)
monthly_spend = np.random.lognormal(mean=3.5, sigma=0.6, size=n).clip(5, 500)
website_visits = np.random.poisson(lam=8, size=n)
support_tickets = np.random.poisson(lam=1.8, size=n)
account_age_months = np.random.exponential(scale=22, size=n).clip(1, 120)
discount_usage = np.random.beta(a=2, b=6, size=n) * 100
satisfaction_score = (
    85 
    - support_tickets * 8 
    + website_visits * 0.5 
    - discount_usage * 0.05
    + np.random.normal(0, 8, n)
).clip(0, 100)

region = np.random.choice(["North", "South", "East", "West"], size=n, p=[0.28, 0.25, 0.25, 0.22])
device = np.random.choice(["Mobile", "Desktop", "Tablet"], size=n, p=[0.58, 0.32, 0.10])
plan = np.random.choice(["Basic", "Standard", "Premium"], size=n, p=[0.42, 0.43, 0.15])
payment_method = np.random.choice(["Card", "Cash", "Wallet", "Bank Transfer"], size=n, p=[0.46, 0.18, 0.25, 0.11])

# Redundant correlated feature
income_duplicate = income * 0.95 + np.random.normal(0, 5000, n)

# Low variance feature
almost_constant = np.random.choice([1, 2], size=n, p=[0.98, 0.02])

# Churn probability
logit = (
    -2.2
    + support_tickets * 0.35
    - satisfaction_score * 0.035
    - account_age_months * 0.012
    - monthly_spend * 0.004
    + discount_usage * 0.008
    + np.where(plan == "Basic", 0.35, 0)
    + np.where(device == "Mobile", 0.10, 0)
    + np.where(payment_method == "Cash", 0.20, 0)
)

churn_probability = 1 / (1 + np.exp(-logit))
churn = np.random.binomial(1, churn_probability)

# Leakage-like feature: created after churn decision in real life
# It is suspicious because it directly depends on churn.
retention_call_made = np.where(churn == 1, np.random.binomial(1, 0.75, n), np.random.binomial(1, 0.08, n))

df = pd.DataFrame({
    "age": age.round(1),
    "income": income.round(0),
    "income_duplicate": income_duplicate.round(0),
    "monthly_spend": monthly_spend.round(2),
    "website_visits": website_visits,
    "support_tickets": support_tickets,
    "account_age_months": account_age_months.round(1),
    "discount_usage": discount_usage.round(1),
    "satisfaction_score": satisfaction_score.round(1),
    "region": region,
    "device": device,
    "plan": plan,
    "payment_method": payment_method,
    "almost_constant": almost_constant,
    "retention_call_made": retention_call_made,
    "churn": churn
})

# Inject missing values
missing_income_idx = np.random.choice(df.index, size=120, replace=False)
missing_satisfaction_idx = np.random.choice(df.index, size=100, replace=False)
missing_region_idx = np.random.choice(df.index, size=80, replace=False)

df.loc[missing_income_idx, "income"] = np.nan
df.loc[missing_satisfaction_idx, "satisfaction_score"] = np.nan
df.loc[missing_region_idx, "region"] = np.nan

# Inject outliers
outlier_idx = np.random.choice(df.index, size=25, replace=False)
df.loc[outlier_idx, "monthly_spend"] = df.loc[outlier_idx, "monthly_spend"] * 8

df.head()

# 5. First inspection

Before engineering features, inspect:

- shape
- data types
- missing values
- target balance
- numeric summary
- categorical summary

In [ ]:
print("Shape:", df.shape)

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nTarget balance:")
display(df["churn"].value_counts(normalize=True).round(3))

print("\nNumeric summary:")
display(df.describe().T.round(2))

print("\nCategorical columns:")
categorical_cols = df.select_dtypes(include="object").columns.tolist()
display(df[categorical_cols].describe())

# 6. Separate features and target

Important:

Always split features and target clearly.

```text
X = input features
y = target
```

Also, split train/test before learning transformations that depend on data.

Why?

Because if preprocessing learns from test data, it leaks information.

In [ ]:
target = "churn"

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean().round(3))
print("Test target rate:", y_test.mean().round(3))

# 7. Missing value imputation

Models cannot usually handle missing values directly.

Common imputation strategies:

## Numeric features

- mean
- median
- constant value
- model-based imputation

## Categorical features

- most frequent
- "Unknown" category

### Statistical thinking

Use median if feature is skewed or has outliers.  
Use mean if feature is roughly symmetric.

In [ ]:
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include="object").columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

missing_summary = X_train.isnull().sum().sort_values(ascending=False)
display(missing_summary[missing_summary > 0])

In [ ]:
# Manual example of imputation

income_median = X_train["income"].median()
region_mode = X_train["region"].mode()[0]

print("Income median from train:", income_median)
print("Region mode from train:", region_mode)

X_train_example = X_train.copy()
X_train_example["income_imputed"] = X_train_example["income"].fillna(income_median)
X_train_example["region_imputed"] = X_train_example["region"].fillna(region_mode)

display(X_train_example[["income", "income_imputed", "region", "region_imputed"]].head(10))

# 8. Add missing indicators

Sometimes missingness itself contains information.

Example:

If income is missing, maybe the customer did not complete profile.

We can create:

```text
income_missing = 1 if income is missing else 0
```

Missing indicators can help models.

In [ ]:
X_train_missing_demo = X_train.copy()

for col in ["income", "satisfaction_score", "region"]:
    X_train_missing_demo[col + "_missing"] = X_train_missing_demo[col].isnull().astype(int)

display(X_train_missing_demo[[
    "income", "income_missing",
    "satisfaction_score", "satisfaction_score_missing",
    "region", "region_missing"
]].head())

# 9. Outlier detection using IQR

Outliers can distort statistics and affect models.

IQR rule:

```text
IQR = Q3 - Q1
Lower = Q1 - 1.5 × IQR
Upper = Q3 + 1.5 × IQR
```

Values outside are possible outliers.

Important:

Outliers are not always wrong.  
They may be rare but valid customers.

In [ ]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return lower, upper

outlier_rows = []

for col in numeric_cols:
    series = X_train[col].dropna()
    lower, upper = iqr_bounds(series)
    outlier_count = ((series < lower) | (series > upper)).sum()

    outlier_rows.append({
        "feature": col,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": outlier_count,
        "outlier_percent": outlier_count / len(series) * 100,
        "skew": series.skew()
    })

outlier_report = pd.DataFrame(outlier_rows).sort_values("outlier_percent", ascending=False)
display(outlier_report.round(2))

# 10. Outlier capping / winsorization

Instead of deleting outliers, we can cap them.

Example:

```text
if value > upper_bound, replace with upper_bound
if value < lower_bound, replace with lower_bound
```

This is called capping or winsorization.

It reduces extreme influence while keeping rows.

In [ ]:
def cap_iqr_train_test(train, test, column):
    lower, upper = iqr_bounds(train[column].dropna())

    train_capped = train[column].clip(lower, upper)
    test_capped = test[column].clip(lower, upper)

    return train_capped, test_capped, lower, upper

train_spend_capped, test_spend_capped, lower_spend, upper_spend = cap_iqr_train_test(
    X_train,
    X_test,
    "monthly_spend"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(X_train["monthly_spend"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Original Monthly Spend")

sns.histplot(train_spend_capped, bins=50, kde=True, ax=axes[1])
axes[1].set_title("Capped Monthly Spend")

plt.tight_layout()
plt.show()

print("Lower cap:", round(lower_spend, 2))
print("Upper cap:", round(upper_spend, 2))

# 11. Percentile capping

Another common approach:

```text
cap lower values at 1st percentile
cap upper values at 99th percentile
```

This is useful for highly skewed business data.

In [ ]:
def percentile_cap_train_test(train, test, column, lower_q=0.01, upper_q=0.99):
    lower = train[column].quantile(lower_q)
    upper = train[column].quantile(upper_q)

    train_capped = train[column].clip(lower, upper)
    test_capped = test[column].clip(lower, upper)

    return train_capped, test_capped, lower, upper

train_income_capped, test_income_capped, lower_income, upper_income = percentile_cap_train_test(
    X_train,
    X_test,
    "income",
    0.01,
    0.99
)

print("Income lower cap:", round(lower_income, 2))
print("Income upper cap:", round(upper_income, 2))

# 12. Skewness and log transform

Right-skewed features have long tails.

Examples:

- income
- transaction amount
- monthly spend
- time on site

Log transform can reduce skewness:

```python
np.log1p(x)
```

`log1p` means log(1 + x), safe when x contains zero.

In [ ]:
skew_values = X_train[numeric_cols].skew().sort_values(ascending=False)
display(skew_values.round(3))

plt.figure(figsize=(9, 5))
sns.barplot(x=skew_values.values, y=skew_values.index)
plt.title("Skewness of Numeric Features")
plt.xlabel("Skewness")
plt.ylabel("Feature")
plt.show()

In [ ]:
X_train_log_demo = X_train.copy()

for col in ["income", "monthly_spend", "account_age_months"]:
    X_train_log_demo["log_" + col] = np.log1p(X_train_log_demo[col])

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for row, col in enumerate(["income", "monthly_spend", "account_age_months"]):
    sns.histplot(X_train_log_demo[col], bins=40, kde=True, ax=axes[row, 0])
    axes[row, 0].set_title(f"Original {col} | skew={X_train_log_demo[col].skew():.2f}")

    sns.histplot(X_train_log_demo["log_" + col], bins=40, kde=True, ax=axes[row, 1])
    axes[row, 1].set_title(f"Log {col} | skew={X_train_log_demo['log_' + col].skew():.2f}")

plt.tight_layout()
plt.show()

# 13. Scaling features

Scaling changes feature values to comparable ranges.

Why scaling matters:

Some models are sensitive to scale:

- Logistic Regression
- SVM
- KNN
- Neural Networks
- PCA
- Gradient descent-based models

Tree-based models are usually less sensitive:

- Decision Tree
- Random Forest
- Gradient Boosting

---

## Common scalers

### StandardScaler
Mean = 0, standard deviation = 1.

### MinMaxScaler
Values between 0 and 1.

### RobustScaler
Uses median and IQR, more robust to outliers.

In [ ]:
scale_demo_cols = ["income", "monthly_spend", "website_visits", "support_tickets"]

# Fill missing for demo
scale_demo = X_train[scale_demo_cols].copy()
scale_demo = scale_demo.fillna(scale_demo.median())

standard_scaled = StandardScaler().fit_transform(scale_demo)
minmax_scaled = MinMaxScaler().fit_transform(scale_demo)
robust_scaled = RobustScaler().fit_transform(scale_demo)

standard_df = pd.DataFrame(standard_scaled, columns=scale_demo_cols)
minmax_df = pd.DataFrame(minmax_scaled, columns=scale_demo_cols)
robust_df = pd.DataFrame(robust_scaled, columns=scale_demo_cols)

print("Original summary:")
display(scale_demo.describe().round(2))

print("Standard scaled summary:")
display(standard_df.describe().round(2))

print("MinMax scaled summary:")
display(minmax_df.describe().round(2))

print("Robust scaled summary:")
display(robust_df.describe().round(2))

# 14. Visual comparison of scalers

Robust scaling is useful when outliers are present.

In [ ]:
feature = "monthly_spend"

plot_df = pd.DataFrame({
    "Original": scale_demo[feature].values,
    "StandardScaler": standard_df[feature].values,
    "MinMaxScaler": minmax_df[feature].values,
    "RobustScaler": robust_df[feature].values
})

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()

for ax, col in zip(axes, plot_df.columns):
    sns.histplot(plot_df[col], bins=40, kde=True, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

# 15. Binning continuous variables

Binning means converting continuous values into groups.

Example:

```text
age → young, middle, senior
```

Binning can help when:

- relationship is non-linear
- we need interpretability
- outliers should be grouped
- business rules use ranges

But binning can also lose information, so use carefully.

In [ ]:
X_train_bin_demo = X_train.copy()

X_train_bin_demo["age_group"] = pd.cut(
    X_train_bin_demo["age"],
    bins=[17, 25, 35, 50, 80],
    labels=["18-25", "26-35", "36-50", "51+"]
)

X_train_bin_demo["spend_group"] = pd.qcut(
    X_train_bin_demo["monthly_spend"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

display(X_train_bin_demo[["age", "age_group", "monthly_spend", "spend_group"]].head())

age_churn = pd.concat([X_train_bin_demo["age_group"], y_train.reset_index(drop=True)], axis=1)

In [ ]:
# Need aligned data for grouped target rate
bin_df = X_train.copy()
bin_df["churn"] = y_train.values
bin_df["age_group"] = pd.cut(
    bin_df["age"],
    bins=[17, 25, 35, 50, 80],
    labels=["18-25", "26-35", "36-50", "51+"]
)

age_group_churn = bin_df.groupby("age_group")["churn"].mean().reset_index()

display(age_group_churn)

plt.figure(figsize=(8, 5))
sns.barplot(data=age_group_churn, x="age_group", y="churn")
plt.title("Churn Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Churn Rate")
plt.show()

# 16. One-hot encoding categorical variables

Machine learning models need numbers.

One-hot encoding creates binary columns.

Example:

```text
plan = Basic, Standard, Premium
```

becomes:

```text
plan_Basic
plan_Standard
plan_Premium
```

This is good for nominal categories with no natural order.

In [ ]:
onehot_demo = pd.get_dummies(X_train[["plan", "device", "payment_method"]].head(10), drop_first=False)

display(X_train[["plan", "device", "payment_method"]].head(10))
display(onehot_demo)

# 17. Target encoding intuition

Target encoding replaces a category with the average target rate for that category.

Example:

```text
plan_Basic → average churn rate for Basic plan
```

This can be powerful, but dangerous.

Why dangerous?

If done before train/test split, it leaks target information.

Correct approach:

- compute encoding only on training data
- apply mapping to validation/test
- use cross-validation target encoding for stronger safety

In [ ]:
target_encoding_demo = X_train.copy()
target_encoding_demo["churn"] = y_train.values

plan_churn_rate = target_encoding_demo.groupby("plan")["churn"].mean()

display(plan_churn_rate)

X_train_encoded_demo = X_train.copy()
X_test_encoded_demo = X_test.copy()

global_churn_rate = y_train.mean()

X_train_encoded_demo["plan_target_encoded"] = X_train_encoded_demo["plan"].map(plan_churn_rate)
X_test_encoded_demo["plan_target_encoded"] = X_test_encoded_demo["plan"].map(plan_churn_rate).fillna(global_churn_rate)

display(X_train_encoded_demo[["plan", "plan_target_encoded"]].head())

# 18. Low-variance feature removal

Features with almost no variation often do not help.

Example:

```text
almost_constant = 1 for 98% of rows
```

A model cannot learn much from a feature that barely changes.

VarianceThreshold removes low-variance numeric features.

In [ ]:
numeric_train_filled = X_train[numeric_cols].fillna(X_train[numeric_cols].median())

variances = numeric_train_filled.var().sort_values()

display(variances.round(5))

plt.figure(figsize=(9, 5))
sns.barplot(x=variances.values, y=variances.index)
plt.title("Feature Variance")
plt.xlabel("Variance")
plt.ylabel("Feature")
plt.show()

In [ ]:
selector = VarianceThreshold(threshold=0.01)
selector.fit(numeric_train_filled)

selected_numeric_cols = numeric_train_filled.columns[selector.get_support()].tolist()
removed_numeric_cols = numeric_train_filled.columns[~selector.get_support()].tolist()

print("Selected numeric columns:", selected_numeric_cols)
print("Removed low-variance columns:", removed_numeric_cols)

# 19. Correlation filtering

Highly correlated features may be redundant.

Example:

```text
income
income_duplicate
```

If two features have correlation above 0.95, one may be removed.

This can help:

- reduce redundancy
- improve interpretability
- reduce multicollinearity for linear models

In [ ]:
corr = numeric_train_filled.corr().abs()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", linewidths=0.5)
plt.title("Absolute Correlation Matrix")
plt.show()

In [ ]:
def find_highly_correlated_features(corr_matrix, threshold=0.90):
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    pairs = []
    to_drop = []

    for col in upper.columns:
        high_corr = upper.index[upper[col] > threshold].tolist()
        for row in high_corr:
            pairs.append((row, col, upper.loc[row, col]))
            to_drop.append(col)

    return pd.DataFrame(pairs, columns=["feature_1", "feature_2", "correlation"]), sorted(set(to_drop))

high_corr_pairs, corr_drop_cols = find_highly_correlated_features(corr, threshold=0.90)

display(high_corr_pairs)
print("Suggested columns to drop:", corr_drop_cols)

# 20. Feature-target relationship

Before feature selection, inspect relationship with target.

For numeric features:

- correlation with target
- distribution by target
- statistical tests
- mutual information

Correlation is useful but limited because it mainly captures linear relationships.

In [ ]:
train_with_target = X_train.copy()
train_with_target["churn"] = y_train.values

numeric_for_target = train_with_target.select_dtypes(include=np.number).columns.tolist()

target_corr = train_with_target[numeric_for_target].corr()["churn"].drop("churn").sort_values(key=lambda s: np.abs(s), ascending=False)

display(target_corr.round(3))

plt.figure(figsize=(9, 5))
sns.barplot(x=target_corr.values, y=target_corr.index)
plt.title("Numeric Feature Correlation with Churn")
plt.xlabel("Correlation")
plt.ylabel("Feature")
plt.show()

# 21. ANOVA F-test feature selection

ANOVA F-test checks whether numeric feature means differ across target classes.

In sklearn:

```python
f_classif
```

Higher score means stronger relationship with target.

Good for numeric features and classification.

In [ ]:
# Prepare numeric data: impute missing values
X_num_train = X_train[numeric_cols].copy()
X_num_train = X_num_train.fillna(X_num_train.median())

f_scores, f_pvalues = f_classif(X_num_train, y_train)

anova_df = pd.DataFrame({
    "feature": numeric_cols,
    "f_score": f_scores,
    "p_value": f_pvalues
}).sort_values("f_score", ascending=False)

display(anova_df.round(5))

plt.figure(figsize=(9, 5))
sns.barplot(data=anova_df, x="f_score", y="feature")
plt.title("ANOVA F-test Scores")
plt.xlabel("F-score")
plt.ylabel("Feature")
plt.show()

# 22. Chi-square feature selection

Chi-square is used for non-negative features and classification.

It is often used for:

- count features
- one-hot encoded categorical features
- text bag-of-words features

Important:

Chi-square requires non-negative input values.

In [ ]:
# For chi-square, use non-negative numeric features
chi_cols = ["age", "income", "monthly_spend", "website_visits", "support_tickets", "account_age_months", "discount_usage", "satisfaction_score", "almost_constant", "retention_call_made"]

X_chi = X_train[chi_cols].copy()
X_chi = X_chi.fillna(X_chi.median())

# Ensure non-negative
X_chi = X_chi.clip(lower=0)

chi_scores, chi_pvalues = chi2(X_chi, y_train)

chi_df = pd.DataFrame({
    "feature": chi_cols,
    "chi2_score": chi_scores,
    "p_value": chi_pvalues
}).sort_values("chi2_score", ascending=False)

display(chi_df.round(5))

plt.figure(figsize=(9, 5))
sns.barplot(data=chi_df, x="chi2_score", y="feature")
plt.title("Chi-square Feature Scores")
plt.xlabel("Chi-square score")
plt.ylabel("Feature")
plt.show()

# 23. Mutual information

Mutual information measures how much knowing a feature reduces uncertainty about the target.

It can capture non-linear relationships better than correlation.

Higher mutual information means stronger dependency.

In [ ]:
mi_scores = mutual_info_classif(X_num_train, y_train, random_state=42)

mi_df = pd.DataFrame({
    "feature": numeric_cols,
    "mutual_information": mi_scores
}).sort_values("mutual_information", ascending=False)

display(mi_df.round(5))

plt.figure(figsize=(9, 5))
sns.barplot(data=mi_df, x="mutual_information", y="feature")
plt.title("Mutual Information Scores")
plt.xlabel("Mutual Information")
plt.ylabel("Feature")
plt.show()

# 24. SelectKBest

`SelectKBest` keeps the top K features based on a scoring function.

Examples:

```python
SelectKBest(score_func=f_classif, k=5)
SelectKBest(score_func=mutual_info_classif, k=5)
```

Feature selection helps:

- reduce noise
- improve speed
- improve interpretability
- reduce overfitting risk

In [ ]:
k = 6

select_k = SelectKBest(score_func=f_classif, k=k)
select_k.fit(X_num_train, y_train)

selected_features = X_num_train.columns[select_k.get_support()].tolist()

print(f"Top {k} numeric features using ANOVA F-test:")
print(selected_features)

# 25. Data leakage

Data leakage happens when training data contains information that would not be available at prediction time.

Leakage can make model performance look excellent during training/testing but fail in real life.

Examples:

- `retention_call_made` if calls happen after churn risk is known
- future payment status
- cancellation date
- target-derived columns
- post-outcome customer service actions

Leakage is dangerous because it creates fake performance.

In [ ]:
# Compare model with leakage feature vs without leakage feature

features_with_leakage = numeric_cols.copy()
features_without_leakage = [col for col in numeric_cols if col != "retention_call_made"]

def evaluate_numeric_features(feature_list):
    X_tr = X_train[feature_list].copy()
    X_te = X_test[feature_list].copy()

    # Fill missing
    medians = X_tr.median()
    X_tr = X_tr.fillna(medians)
    X_te = X_te.fillna(medians)

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ])

    pipeline.fit(X_tr, y_train)
    pred = pipeline.predict(X_te)
    proba = pipeline.predict_proba(X_te)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba)
    }

with_leakage_result = evaluate_numeric_features(features_with_leakage)
without_leakage_result = evaluate_numeric_features(features_without_leakage)

leakage_compare = pd.DataFrame([
    {"feature_set": "With leakage feature", **with_leakage_result},
    {"feature_set": "Without leakage feature", **without_leakage_result}
])

display(leakage_compare.round(3))

# 26. Leakage detection checklist

Ask these questions:

1. Would this feature be available at prediction time?
2. Was this feature created after the target event?
3. Does this feature directly depend on the target?
4. Is the feature suspiciously predictive?
5. Does model performance become unrealistically high with this feature?
6. Is the feature a future value?
7. Is the feature an ID that encodes target information?
8. Was preprocessing done before train/test split?

If yes, investigate leakage.

# 27. Feature interactions

Sometimes features are useful together.

Examples:

```text
monthly_spend / income
support_tickets / account_age
visits × satisfaction
```

Interactions can help linear models capture more complex patterns.

But too many interactions can cause overfitting.

In [ ]:
X_train_interact = X_train.copy()
X_test_interact = X_test.copy()

# Create interaction features safely
X_train_interact["spend_to_income_ratio"] = X_train_interact["monthly_spend"] / (X_train_interact["income"] + 1)
X_test_interact["spend_to_income_ratio"] = X_test_interact["monthly_spend"] / (X_test_interact["income"] + 1)

X_train_interact["tickets_per_account_month"] = X_train_interact["support_tickets"] / (X_train_interact["account_age_months"] + 1)
X_test_interact["tickets_per_account_month"] = X_test_interact["support_tickets"] / (X_test_interact["account_age_months"] + 1)

X_train_interact["visits_x_satisfaction"] = X_train_interact["website_visits"] * X_train_interact["satisfaction_score"]
X_test_interact["visits_x_satisfaction"] = X_test_interact["website_visits"] * X_test_interact["satisfaction_score"]

display(X_train_interact[[
    "spend_to_income_ratio",
    "tickets_per_account_month",
    "visits_x_satisfaction"
]].head())

# 28. Building a manual engineered dataset

Now we create a clean engineered dataset manually.

Steps:

1. remove leakage feature  
2. impute missing values  
3. add missing indicators  
4. cap outliers  
5. log transform skewed features  
6. create interactions  
7. one-hot encode categories  
8. scale numeric features

In [ ]:
def manual_feature_engineering_fit_transform(X_train, X_test):
    X_train_fe = X_train.copy()
    X_test_fe = X_test.copy()

    # 1. Remove leakage
    leakage_cols = ["retention_call_made"]
    X_train_fe = X_train_fe.drop(columns=leakage_cols, errors="ignore")
    X_test_fe = X_test_fe.drop(columns=leakage_cols, errors="ignore")

    # 2. Add missing indicators
    missing_indicator_cols = ["income", "satisfaction_score", "region"]

    for col in missing_indicator_cols:
        X_train_fe[col + "_missing"] = X_train_fe[col].isnull().astype(int)
        X_test_fe[col + "_missing"] = X_test_fe[col].isnull().astype(int)

    # 3. Identify numeric and categorical columns
    num_cols = X_train_fe.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_train_fe.select_dtypes(include="object").columns.tolist()

    # 4. Impute numeric
    medians = X_train_fe[num_cols].median()
    X_train_fe[num_cols] = X_train_fe[num_cols].fillna(medians)
    X_test_fe[num_cols] = X_test_fe[num_cols].fillna(medians)

    # 5. Impute categorical
    modes = {}
    for col in cat_cols:
        modes[col] = X_train_fe[col].mode()[0]
        X_train_fe[col] = X_train_fe[col].fillna(modes[col])
        X_test_fe[col] = X_test_fe[col].fillna(modes[col])

    # 6. Cap outliers using percentile caps learned from train
    cap_cols = ["income", "monthly_spend", "account_age_months"]

    for col in cap_cols:
        lower = X_train_fe[col].quantile(0.01)
        upper = X_train_fe[col].quantile(0.99)
        X_train_fe[col] = X_train_fe[col].clip(lower, upper)
        X_test_fe[col] = X_test_fe[col].clip(lower, upper)

    # 7. Log transforms
    for col in ["income", "monthly_spend", "account_age_months"]:
        X_train_fe["log_" + col] = np.log1p(X_train_fe[col])
        X_test_fe["log_" + col] = np.log1p(X_test_fe[col])

    # 8. Interaction features
    X_train_fe["spend_to_income_ratio"] = X_train_fe["monthly_spend"] / (X_train_fe["income"] + 1)
    X_test_fe["spend_to_income_ratio"] = X_test_fe["monthly_spend"] / (X_test_fe["income"] + 1)

    X_train_fe["tickets_per_account_month"] = X_train_fe["support_tickets"] / (X_train_fe["account_age_months"] + 1)
    X_test_fe["tickets_per_account_month"] = X_test_fe["support_tickets"] / (X_test_fe["account_age_months"] + 1)

    X_train_fe["visits_x_satisfaction"] = X_train_fe["website_visits"] * X_train_fe["satisfaction_score"]
    X_test_fe["visits_x_satisfaction"] = X_test_fe["website_visits"] * X_test_fe["satisfaction_score"]

    # 9. Drop redundant original columns if log version exists
    X_train_fe = X_train_fe.drop(columns=["income", "monthly_spend", "account_age_months", "income_duplicate"], errors="ignore")
    X_test_fe = X_test_fe.drop(columns=["income", "monthly_spend", "account_age_months", "income_duplicate"], errors="ignore")

    # 10. One-hot encode
    X_train_fe = pd.get_dummies(X_train_fe, drop_first=True)
    X_test_fe = pd.get_dummies(X_test_fe, drop_first=True)

    # Align train and test columns
    X_train_fe, X_test_fe = X_train_fe.align(X_test_fe, join="left", axis=1, fill_value=0)

    # 11. Scale all columns
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fe)
    X_test_scaled = scaler.transform(X_test_fe)

    X_train_final = pd.DataFrame(X_train_scaled, columns=X_train_fe.columns, index=X_train_fe.index)
    X_test_final = pd.DataFrame(X_test_scaled, columns=X_test_fe.columns, index=X_test_fe.index)

    return X_train_final, X_test_final

X_train_fe, X_test_fe = manual_feature_engineering_fit_transform(X_train, X_test)

print("Original train shape:", X_train.shape)
print("Engineered train shape:", X_train_fe.shape)

display(X_train_fe.head())

# 29. Compare raw vs engineered features

Now we compare model performance.

We use Logistic Regression because it is sensitive to scaling and feature engineering.

In [ ]:
# Raw numeric baseline without leakage
raw_feature_list = [col for col in numeric_cols if col != "retention_call_made"]

X_train_raw = X_train[raw_feature_list].copy()
X_test_raw = X_test[raw_feature_list].copy()

raw_medians = X_train_raw.median()
X_train_raw = X_train_raw.fillna(raw_medians)
X_test_raw = X_test_raw.fillna(raw_medians)

raw_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

raw_pipeline.fit(X_train_raw, y_train)
raw_pred = raw_pipeline.predict(X_test_raw)
raw_proba = raw_pipeline.predict_proba(X_test_raw)[:, 1]

# Engineered model
fe_model = LogisticRegression(max_iter=1000)
fe_model.fit(X_train_fe, y_train)
fe_pred = fe_model.predict(X_test_fe)
fe_proba = fe_model.predict_proba(X_test_fe)[:, 1]

comparison = pd.DataFrame([
    {
        "feature_set": "Raw numeric only",
        "accuracy": accuracy_score(y_test, raw_pred),
        "precision": precision_score(y_test, raw_pred, zero_division=0),
        "recall": recall_score(y_test, raw_pred, zero_division=0),
        "f1": f1_score(y_test, raw_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, raw_proba)
    },
    {
        "feature_set": "Engineered features",
        "accuracy": accuracy_score(y_test, fe_pred),
        "precision": precision_score(y_test, fe_pred, zero_division=0),
        "recall": recall_score(y_test, fe_pred, zero_division=0),
        "f1": f1_score(y_test, fe_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, fe_proba)
    }
])

display(comparison.round(3))

# 30. Visual comparison

Feature engineering should be judged by metrics and error behavior.

In [ ]:
comparison_long = comparison.melt(
    id_vars="feature_set",
    value_vars=["accuracy", "precision", "recall", "f1", "roc_auc"],
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(10, 5))
sns.barplot(data=comparison_long, x="metric", y="score", hue="feature_set")
plt.title("Raw vs Engineered Feature Performance")
plt.xlabel("Metric")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.legend(title="Feature Set")
plt.show()

# 31. Confusion matrix after feature engineering

Always inspect false positives and false negatives.

In [ ]:
cm = confusion_matrix(y_test, fe_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred 0", "Pred 1"],
    yticklabels=["Actual 0", "Actual 1"]
)
plt.title("Confusion Matrix — Engineered Features")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print(classification_report(y_test, fe_pred, zero_division=0))

# 32. Feature importance from Logistic Regression

For scaled features, Logistic Regression coefficients can give direction and strength.

Positive coefficient:

```text
feature increases probability of churn
```

Negative coefficient:

```text
feature decreases probability of churn
```

Important:

Coefficients are not always causal explanations.

In [ ]:
coef_df = pd.DataFrame({
    "feature": X_train_fe.columns,
    "coefficient": fe_model.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

display(coef_df.head(15))

plt.figure(figsize=(10, 7))
sns.barplot(data=coef_df.head(15), x="coefficient", y="feature")
plt.title("Top Logistic Regression Coefficients")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.show()

# 33. Pipeline-based preprocessing

Manual feature engineering is useful for learning.

But in production, use pipelines.

Pipelines prevent data leakage because transformations are learned only from training data during fit.

We will create:

- numeric pipeline
- categorical pipeline
- column transformer
- full model pipeline

In [ ]:
# Use non-leakage columns
X_no_leak = X.drop(columns=["retention_call_made"], errors="ignore")

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_no_leak,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

numeric_features = X_train_p.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train_p.select_dtypes(include="object").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Handle scikit-learn versions with sparse_output vs sparse
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", encoder)
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

full_pipeline.fit(X_train_p, y_train_p)

pipeline_pred = full_pipeline.predict(X_test_p)
pipeline_proba = full_pipeline.predict_proba(X_test_p)[:, 1]

pipeline_result = {
    "accuracy": accuracy_score(y_test_p, pipeline_pred),
    "precision": precision_score(y_test_p, pipeline_pred, zero_division=0),
    "recall": recall_score(y_test_p, pipeline_pred, zero_division=0),
    "f1": f1_score(y_test_p, pipeline_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test_p, pipeline_proba)
}

display(pd.DataFrame([pipeline_result]).round(3))

# 34. Add feature selection to pipeline

We can add `SelectKBest` after preprocessing.

This keeps only top features.

Because one-hot encoding creates many columns, feature selection can reduce dimensionality.

In [ ]:
feature_selection_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("select", SelectKBest(score_func=f_classif, k=12)),
    ("model", LogisticRegression(max_iter=1000))
])

feature_selection_pipeline.fit(X_train_p, y_train_p)

fs_pred = feature_selection_pipeline.predict(X_test_p)
fs_proba = feature_selection_pipeline.predict_proba(X_test_p)[:, 1]

fs_result = {
    "accuracy": accuracy_score(y_test_p, fs_pred),
    "precision": precision_score(y_test_p, fs_pred, zero_division=0),
    "recall": recall_score(y_test_p, fs_pred, zero_division=0),
    "f1": f1_score(y_test_p, fs_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test_p, fs_proba)
}

display(pd.DataFrame([
    {"pipeline": "All features", **pipeline_result},
    {"pipeline": "SelectKBest top 12", **fs_result}
]).round(3))

# 35. Cross-validation with pipelines

The correct way to evaluate preprocessing is to put preprocessing inside the pipeline and cross-validate the whole pipeline.

Wrong:

```text
scale whole dataset first, then cross validate
```

Correct:

```text
Pipeline(preprocessing + model), then cross validate
```

This avoids leakage.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipeline_scores = cross_val_score(
    full_pipeline,
    X_no_leak,
    y,
    cv=cv,
    scoring="f1"
)

fs_pipeline_scores = cross_val_score(
    feature_selection_pipeline,
    X_no_leak,
    y,
    cv=cv,
    scoring="f1"
)

cv_summary = pd.DataFrame({
    "pipeline": ["All features", "SelectKBest"],
    "mean_f1": [pipeline_scores.mean(), fs_pipeline_scores.mean()],
    "std_f1": [pipeline_scores.std(), fs_pipeline_scores.std()]
})

display(cv_summary.round(3))

plt.figure(figsize=(8, 5))
sns.boxplot(data=pd.DataFrame({
    "All features": pipeline_scores,
    "SelectKBest": fs_pipeline_scores
}))
plt.title("Cross-Validation F1 Scores")
plt.ylabel("F1 Score")
plt.show()

# 36. Tree models and feature engineering

Tree models usually need less scaling.

Random Forest can handle:

- non-linear relationships
- interactions
- different feature scales

But it still needs:

- missing value handling
- categorical encoding
- leakage prevention

Feature engineering can still help, but scaling is less important for trees.

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=250, max_depth=8, random_state=42))
])

rf_pipeline.fit(X_train_p, y_train_p)

rf_pred = rf_pipeline.predict(X_test_p)
rf_proba = rf_pipeline.predict_proba(X_test_p)[:, 1]

rf_result = {
    "accuracy": accuracy_score(y_test_p, rf_pred),
    "precision": precision_score(y_test_p, rf_pred, zero_division=0),
    "recall": recall_score(y_test_p, rf_pred, zero_division=0),
    "f1": f1_score(y_test_p, rf_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test_p, rf_proba)
}

display(pd.DataFrame([
    {"model": "Logistic Regression Pipeline", **pipeline_result},
    {"model": "Random Forest Pipeline", **rf_result}
]).round(3))

# 37. Interactive transformation explorer

Choose a feature and transformation.

This helps build intuition about how preprocessing changes distributions.

In [ ]:
def transformation_explorer(feature="monthly_spend", transform="log1p"):
    data = X_train[feature].copy()

    if data.isnull().any():
        data = data.fillna(data.median())

    if transform == "original":
        transformed = data
    elif transform == "log1p":
        transformed = np.log1p(data.clip(lower=0))
    elif transform == "standard":
        transformed = pd.Series(StandardScaler().fit_transform(data.values.reshape(-1, 1)).ravel())
    elif transform == "minmax":
        transformed = pd.Series(MinMaxScaler().fit_transform(data.values.reshape(-1, 1)).ravel())
    elif transform == "robust":
        transformed = pd.Series(RobustScaler().fit_transform(data.values.reshape(-1, 1)).ravel())
    elif transform == "iqr_capped":
        lower, upper = iqr_bounds(data)
        transformed = data.clip(lower, upper)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.histplot(data, bins=40, kde=True, ax=axes[0])
    axes[0].set_title(f"Original {feature}\nSkew={pd.Series(data).skew():.2f}")

    sns.histplot(transformed, bins=40, kde=True, ax=axes[1])
    axes[1].set_title(f"{transform} {feature}\nSkew={pd.Series(transformed).skew():.2f}")

    plt.tight_layout()
    plt.show()

    print("Original mean:", round(pd.Series(data).mean(), 3))
    print("Transformed mean:", round(pd.Series(transformed).mean(), 3))
    print("Original std:", round(pd.Series(data).std(), 3))
    print("Transformed std:", round(pd.Series(transformed).std(), 3))

interactive_numeric_features = ["income", "monthly_spend", "account_age_months", "discount_usage", "satisfaction_score", "support_tickets", "website_visits"]

if WIDGETS_AVAILABLE:
    interact(
        transformation_explorer,
        feature=widgets.Dropdown(options=interactive_numeric_features, value="monthly_spend"),
        transform=widgets.Dropdown(options=["original", "log1p", "standard", "minmax", "robust", "iqr_capped"], value="log1p")
    )
else:
    transformation_explorer("monthly_spend", "log1p")

# 38. Feature engineering decision guide

Use this guide:

| Problem found | Action |
|---|---|
| Missing numeric values | median imputation |
| Missing categorical values | most frequent or Unknown |
| Missingness meaningful | add missing indicator |
| Right-skewed positive feature | log1p transform |
| Strong outliers | cap, robust scale, investigate |
| Different scales | standardize or normalize |
| Low variance | remove |
| High correlation | remove one redundant feature |
| Categorical nominal | one-hot encode |
| High-cardinality category | target encoding carefully |
| Count feature | keep count or log1p |
| Ratio meaningful | create ratio |
| Nonlinear relationship | binning or tree model |
| Suspiciously strong feature | check leakage |

# 39. Mini project — Complete Statistical Feature Engineering Pipeline

Your task is to create a full preprocessing pipeline for this dataset.

## Required sections

### Section 1 — Data inspection
- shape
- missing values
- data types
- target balance

### Section 2 — Statistical profile
- mean, median, std
- skewness
- outlier report
- correlation matrix

### Section 3 — Missing values
- numeric imputation
- categorical imputation
- missing indicators

### Section 4 — Outliers
- IQR report
- percentile capping
- before/after visualization

### Section 5 — Transformations
- log transform skewed features
- scaling comparison
- choose scaler

### Section 6 — Categorical encoding
- one-hot encoding
- target encoding explanation

### Section 7 — Feature selection
- low variance removal
- correlation filtering
- ANOVA F-test
- Chi-square
- mutual information

### Section 8 — Leakage detection
- identify suspicious columns
- compare with/without leakage

### Section 9 — Modeling
- train baseline model
- train engineered model
- train pipeline model
- compare metrics

### Section 10 — Final recommendation
Write at least 10 insights and explain the final preprocessing pipeline.

In [ ]:
# Mini project starter

print("Dataset shape:", df.shape)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nTarget balance:")
display(df["churn"].value_counts(normalize=True).round(3))

print("\nNumeric skewness:")
display(df.select_dtypes(include=np.number).skew().sort_values(ascending=False).round(3))

print("\nHigh correlation pairs:")
display(high_corr_pairs)

print("\nModel comparison:")
display(comparison.round(3))

# 40. Common feature engineering mistakes

Avoid these mistakes:

## Mistake 1: Preprocessing before train/test split
This leaks test information.

## Mistake 2: Scaling categorical encoded IDs
If numbers are labels, not quantities, do not treat them as continuous.

## Mistake 3: Using target encoding incorrectly
Target encoding must be learned only from training data.

## Mistake 4: Keeping leakage features
Suspiciously strong features must be investigated.

## Mistake 5: Removing all outliers blindly
Outliers may be valid and important.

## Mistake 6: Applying log to negative values
Use log only for positive data, or shift carefully.

## Mistake 7: Scaling tree models unnecessarily
Not harmful always, but usually not needed.

## Mistake 8: Dropping correlated features blindly
Keep the feature that is more interpretable or more useful.

## Mistake 9: Feature engineering without validation
Every transformation should be tested.

## Mistake 10: Too many features without reason
Extra noisy features can hurt generalization.

# 41. Practice questions

Answer these in your own words.

1. What is feature engineering?
2. Why does statistics guide feature engineering?
3. Why should preprocessing be fitted only on training data?
4. When should we use mean imputation?
5. When should we use median imputation?
6. Why add missing indicators?
7. What is outlier capping?
8. Difference between IQR capping and percentile capping?
9. Why does log transform help right-skewed features?
10. What is StandardScaler?
11. What is MinMaxScaler?
12. What is RobustScaler?
13. Why is RobustScaler useful with outliers?
14. What is binning?
15. What can be lost when we bin a feature?
16. What is one-hot encoding?
17. What is target encoding?
18. Why can target encoding leak information?
19. What is low-variance feature removal?
20. Why remove highly correlated features?
21. What does ANOVA F-test measure?
22. When is Chi-square feature selection useful?
23. What does mutual information measure?
24. What is data leakage?
25. Why are pipelines important?
26. Why should preprocessing be cross-validated inside the pipeline?
27. Why are tree models less sensitive to scaling?
28. How do you decide whether feature engineering improved a model?

# 42. What we learned

In this notebook, we learned:

- Feature engineering transforms raw data into better model inputs
- Statistics tells us which transformations make sense
- Missing values can be imputed and flagged
- Outliers can be detected using IQR or percentiles
- Capping reduces extreme influence
- Log transform reduces right skew
- Scaling matters for distance-based and gradient-based models
- Robust scaling helps with outliers
- Binning can make nonlinear patterns easier to model
- One-hot encoding handles nominal categories
- Target encoding is powerful but leakage-prone
- Low-variance features often add little value
- Correlation filtering removes redundancy
- ANOVA, Chi-square, and mutual information help select features
- Data leakage can create fake model performance
- Feature interactions can improve linear models
- Pipelines prevent preprocessing leakage
- Cross-validation should include preprocessing steps
- Feature engineering must be evaluated using model metrics

---

# Statistics for Machine Learning Series Complete

We completed 6 notebooks:

1. Descriptive Statistics and Data Understanding  
2. Probability for Machine Learning  
3. Probability Distributions for ML  
4. Sampling, Confidence Intervals, and Hypothesis Testing  
5. Statistical Thinking for Model Evaluation  
6. Feature Engineering Using Statistics  

Now you have the statistical foundation needed for serious machine learning.